https://arxiv.org/html/2409.02292v1
RAMBO: Leaking Secrets from Air-Gap Computers by Spelling Covert Radio Signals from Computer RAM
Mordechai Guri

The RAMBO attack signal generation with OOK modulation

In [ ]:
Algorithm 1 modulateOOK (bits, bitTimeMillis, frameSizeBits)
1: bitEndTime $\leftarrow$ getCurrentTimeMillis()
2: frameStart $\leftarrow$ 0
3: while frameStart $<$ (len(bits) - frameSizeBits) do
4: frameBits $\leftarrow$ [1, 0, 1, 0, 1, 0]
5: frameBits.append(bits[frameStart : frameStart + frameSizeBits])
6: frameStart $+=$ frameSizeBits
7: frameBits.append(calcParity(frameBits))
8: for bit in frameBits do
9: bitEndTime $\leftarrow$ bitEndTime $+$ bitTimeMillis
10: if bit $== 1$ then
11: while getCurrentTimeMillis() $<$ bitEndTime do
12: movnti(memoryAddress, value)
13: end while
14: else
15: sleep(bitEndTime - getCurrentTimeMillis())
16: end if
17: end for
18: end while

The transmission with Manchester encoding

In [ ]:
Algorithm 2 modulateManchester (bits, bitTimeMillis, frameSizeBits)
1: bitEndTime $\leftarrow$ getCurrentTimeMillis()
2: frameStart $\leftarrow$ 0
3: while frameStart $<$ (len(bits) - frameSizeBits) do
4: frameBits $\leftarrow$ [1, 0, 1, 0, 1, 0]
5: frameBits.append(bits[frameStart : frameStart + frameSizeBits])
6: frameStart $+=$ frameSizeBits
7: frameBits.append(calcParity(frameBits))
8: for bit in frameBits do
9: if bit $== 1$ then
10: bitEndTime $\leftarrow$ bitEndTime $+$ bitTimeMillis/2
11: while getCurrentTimeMillis() $<$ bitEndTime do
12: movnti(memoryAddress, value)
13: end while
14: bitEndTime $\leftarrow$ bitEndTime $+$ bitTimeMillis/2
15: sleep(bitEndTime $-$ getCurrentTimeMillis())
16: else
17: bitEndTime $\leftarrow$ bitEndTime $+$ bitTimeMillis/2
18: sleep(bitEndTime $-$ getCurrentTimeMillis())
19: bitEndTime $\leftarrow$ bitEndTime $+$ bitTimeMillis/2
20: while getCurrentTimeMillis() $<$ bitEndTime do
21: movnti(memoryAddress, value)
22: end while
23: end if
24: end for
25: end while

The demodulation algorithm

In [ ]:
Algorithm 3 demodulate(sampleRate, frameSizeBits, bitTime, windowSize, signalRelativeFreq)
1: windowsPerBit $\leftarrow$ sampleRate $/ 1e6 *$ bitTime / windowSize
2:
3: while True do
4: enabled $\leftarrow$ False
5: bitCounter $\leftarrow$ 0
6: bits $\leftarrow$ []
7: while len(bits) $<$ frameSizeBits $+ 1$ do
8: windows.append(getNextWindows(windowSize))
9: for window in windows do
10: spectrum $\leftarrow$ welch(window, windowSize)
11: sample $\leftarrow$ spectrum[signalRelativeFreq]
12: samples.append(sample)
13: end for
14: if not enabled then
15: thresh, enabled $\leftarrow$ detectEnable(samples, windowsPerBit)
16: end if
17: while enabled and len(samples) $>=$ windowsPerBit do
18: bitsToBit(samples, windowsPerBit, thresh)
19: bits.append(bit)
20: if len(bits) $==$ frameSizeBits $+1$ then
21: outputFrame(bits[:-1])
22: if bits[-1] $!=$ calcParity(bits[:-1]) then
23: logParityError()
24: end if
25: end if
26: end while
27: end while
28: end while